# Pre-processing pipeline data - datasetgeneratie voor unsupervised anomaliedetectie

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`  

## Doel

Deze notebook zet _exports_ van het GBS-platform (Johnson Controls) om naar _consistente_, _reproduceerbare_ **trainingsdata** voor een _unsupervised anomaliedetectiemodel_ voor hybride HVAC-systemen. De pipeline bevat `inlezen`, `opschonen`, `normaliseren`, `feature-engineering` en `export` naar een verwerkbaar bestandsformaat.

### Verwachte brondata

- Lijst met object-id's uit Johnson Controls die via de API de brondata op kan vragen. 
- Start- en einddatum in te lezen data.

De lijst ziet er als volgt uit:

```
id,func_datapunt_nr,datapunt_naam,object_id_gbs
1,44,temperatuur zone 1,11.02 DUNANT 1 - Kantoren links.Temperatuur,ba298cd6-2e20-5d5e-8b23-dd6f36801293
2,44,temperatuur zone 1,11.02 DUNANT 1 -Kantoren links.Temperatuur,b70bb261-c83d-5216-beed-0ef6ed5dc2fe
```

### Stappen

1. Inlezen van de csv met object-id's GBS
2. Data ophalen en opslaan via API uit GBS voor gevraagde tijdsperiode van alle id's
3. Opschonen en valideren van de opgehaalde data
4. Samenvoegen van geaggregeerde datapunten tot de functionele datapunten
5. Ophalen weerdata (KMI AWS 10 minuten) gevraagde punten
6. Datapunten synchroniseren tot 10-minuten intervallen en samenvoegen tot 1 dataset
7. Wegschrijven van de samengevoegde dataset

### Gebruik

- Invullen van CSV met object-id's voor het doelgebouw en plaatsen in de map `../data/bronnen.csv`
- Plaatsen van CSV met kmi aws data met daarin de gevraagde periode in `../data/raw/weerdata.csv`
- Doorlopen van de stappen en controleren van uitkomsten
- Bestanden komen terecht in `../data/pre-processed`

In [1]:
import pandas as pd
import requests
import urllib3
from datetime import datetime, date, time, timedelta, timezone
from dotenv import load_dotenv
import os
import re
import numpy as np

## VARIABELEN

In [2]:
# Tijdsvariabelen
START_DATUM = datetime(2026, 3, 9, 0, 0, 0, tzinfo=timezone.utc) # vul in jaar, maand, dag, uur, minuten, seconden
EIND_DATUM  = datetime(2026, 5, 1, 23, 59, 59, tzinfo=timezone.utc)
GEBOUWNAAM = 'dunant1'

In [3]:
# Bestanden
BRON_CSV = f'bronbestanden/bronnen-{GEBOUWNAAM}.csv'

In [4]:
# Laad het .env bestand
load_dotenv('../.env', override=True)

True

## STAP 1 - inladen van brondata

In [5]:
datapunten = pd.read_csv(BRON_CSV, index_col='id')
datapunten.head()

,func_dp_nr,func_dp_naam,func_systeem,gbs_naam,object-id_gbs,opmerking
id,,,,,,
1,1,totaal warmtegebruik warm,warmtepomp(en),11.02 DUNANT 1 - ENERGIEMETER SECUNDAIR WARMTE...,6404863b-d6f6-530b-a8b4-345759d38c36,NaN
2,2,totaal debiet,warmtepomp(en),11.02 DUNANT 1 - ENERGIEMETER SECUNDAIR WARMTE...,971e304d-e5b4-5c9c-b279-b5deb6d0477e,NaN
3,3,status,warmtepomp(en),11.02 DUNANT 1 - Warmtepomp.HVAC.11.02.090.002...,dbccce07-3909-5845-8635-ffef49677026,NaN
4,4,temperatuur vertrek,warmtepomp(en),11.02 DUNANT 1 - Warmtepomp.Intrede warme kant...,861ce3ab-3969-5acd-9329-b0f061a92f34,NaN
5,5,gevraagde temperatuur,warmtepomp(en),11.02 DUNANT 1 - Warmteproduktie.Berekend setp...,dfeaf4ae4-c4a1-508a-bd4a-1d7844a422df,NaN


## STAP 2 - Data ophalen uit GBS

In [6]:
# Schakel SSL-waarschuwingen uit (vaak nodig bij self-signed certificaten van Metasys servers)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [7]:
# --- Configuratie ---
base_url = os.getenv('BASE_URL')
username = os.getenv('USERNAME')
password = os.getenv('PASSWORD')

### Inloggen API JC

In [8]:
# --- 1. Inloggen op de API ---
login_url = f"{base_url}/login"
login_response = requests.post(login_url, json={"username": username, "password": password}, verify=False)
login_response.raise_for_status()

access_token = login_response.json().get("accessToken")
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}
print("Succesvol ingelogd!")

Succesvol ingelogd!


### Ophaal functies

hieronder kan je de functies vinden waarmee de data opgehaald kan worden op de juiste manier.

In [9]:
def haal_meetwaarden_op(object_id_gbs, start_dt, eind_dt, headers):
    """Haalt alle tijdreeks-data op voor één uniek Johnson Controls object."""
    # 1. URL van de samples ophalen
    trend_url = f"{base_url}/timeSeries/{object_id_gbs}/trendedAttributes"
    res = requests.get(trend_url, headers=headers, verify=False)
    
    if res.status_code != 200 or not res.json().get("items"):
        return []
        
    base_samples_url = res.json()["items"][0]["samplesUrl"].split('?')[0]
    
    # 2. Instellingen voor ophalen (zet datetime objecten om naar string)
    params = {
        "startSampleTime": start_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "endSampleTime": eind_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "pageSize": 1000,
        "page": 1,
        "sort": "sampleTime"
    }
    
    # 3. Paginatie: blijf data ophalen zolang er volgende pagina's zijn
    alle_samples = []
    while True:
        resp = requests.get(base_samples_url, headers=headers, params=params, verify=False)
        if resp.status_code != 200:
            break
            
        data = resp.json()
        alle_samples.extend(data.get("items", []))
        
        if not data.get("next"):
            break
        params["page"] += 1
        
    return alle_samples

### Loop ophalen

In [10]:
verzamelde_data = []

# Loop over je de ingelezen CSV (met 'id' als index)
for id_csv, row in datapunten.iterrows():
    object_id_gbs = row['object-id_gbs']
    func_nr = row['func_dp_nr']
    naam = row['func_dp_naam']
    systeem = row['func_systeem']
    
    print(f"Ophalen data voor ID {id_csv}: {naam}...")
    
    # Haal ruwe data op via API
    samples = haal_meetwaarden_op(object_id_gbs, START_DATUM, EIND_DATUM, headers)
    
    # Voeg jouw extra csv informatie toe aan elk datapunt (Maken van het Flat-bestand)
    for sample in samples:
        verzamelde_data.append({
            'timestamp': sample.get('sampleTime'),
            'id': id_csv,
            'func_dp_nr': func_nr,
            'datapunt_naam': naam,
            'datapunt_systeem': systeem,
            'waarde': sample.get('value')
        })

# Maak het Long-DataFrame
df_long = pd.DataFrame(verzamelde_data)
# Zorg dat pandas snapt dat kolom 'timestamp' echt een datum/tijd is
df_long['timestamp'] = pd.to_datetime(df_long['timestamp'])

print(f"\nKlaar! {len(df_long)} metingen opgehaald.")
display(df_long.head())

Ophalen data voor ID 1: totaal warmtegebruik warm...
Ophalen data voor ID 2: totaal debiet...
Ophalen data voor ID 3: status...
Ophalen data voor ID 4: temperatuur vertrek...
Ophalen data voor ID 5: gevraagde temperatuur...
Ophalen data voor ID 6: temperatuur retour...
Ophalen data voor ID 7: status pomp...
Ophalen data voor ID 8: temperatuur vertrek...
Ophalen data voor ID 9: gevraagde temperatuur...
Ophalen data voor ID 10: status...
Ophalen data voor ID 11: temperatuur retour...
Ophalen data voor ID 12: status pomp...
Ophalen data voor ID 13: totaal warmtegebruik warm injectie...
Ophalen data voor ID 14: totaal debiet injectie...
Ophalen data voor ID 15: stand klep injectie...
Ophalen data voor ID 16: totaal warmtegebruik warm...
Ophalen data voor ID 17: totaal warmtegebruik warm...
Ophalen data voor ID 18: totaal debiet...
Ophalen data voor ID 19: totaal debiet...
Ophalen data voor ID 20: temperatuur vertrek...
Ophalen data voor ID 21: temperatuur retour...
Ophalen data voor ID 22:

,timestamp,id,func_dp_nr,datapunt_naam,datapunt_systeem,waarde
0,2026-03-09 00:00:00+00:00,1,1,totaal warmtegebruik warm,warmtepomp(en),186513.0
1,2026-03-09 00:10:00+00:00,1,1,totaal warmtegebruik warm,warmtepomp(en),186520.0
2,2026-03-09 00:20:00+00:00,1,1,totaal warmtegebruik warm,warmtepomp(en),186529.0
3,2026-03-09 00:30:00+00:00,1,1,totaal warmtegebruik warm,warmtepomp(en),186537.0
4,2026-03-09 00:40:00+00:00,1,1,totaal warmtegebruik warm,warmtepomp(en),186544.0


#### Zet alle waarden om naar numerieke waarden

Sommige statussen komen binnen als `offonEnumSet.0` of `offonEnumSet.1`. Deze zetten we om naar $0$ of $1$.

In [11]:
# bewaar origineel
if 'waarde_raw' not in df_long.columns:
    df_long['waarde_raw'] = df_long['waarde']

# normaliseer en parseer
vals_clean = df_long['waarde_raw'].astype(str).str.strip().str.replace(',', '.', regex=False)
parsed = pd.to_numeric(vals_clean.str.rstrip('%'), errors='coerce')

# voor resterende niet-parseerbare strings: haal laatste cijfer
mask_missing = parsed.isna()
extracted = vals_clean.str.extract(r'(\d+)$')[0]
parsed.loc[mask_missing] = pd.to_numeric(extracted.loc[mask_missing], errors='coerce')

# zet als finale numerieke kolom terug (Int64 indien geheel)
df_long['waarde'] = parsed
if df_long['waarde'].dropna().apply(float.is_integer).all():
    df_long['waarde'] = df_long['waarde'].astype('Int64')

# checks
print("Totaal:", len(df_long), "Niet-NaN:", df_long['waarde'].notna().sum(), "NaN:", df_long['waarde'].isna().sum())

Totaal: 525304 Niet-NaN: 525304 NaN: 0


In [12]:
print(df_long.head())

                  timestamp  id  func_dp_nr              datapunt_naam  \
0 2026-03-09 00:00:00+00:00   1           1  totaal warmtegebruik warm   
1 2026-03-09 00:10:00+00:00   1           1  totaal warmtegebruik warm   
2 2026-03-09 00:20:00+00:00   1           1  totaal warmtegebruik warm   
3 2026-03-09 00:30:00+00:00   1           1  totaal warmtegebruik warm   
4 2026-03-09 00:40:00+00:00   1           1  totaal warmtegebruik warm   

  datapunt_systeem    waarde waarde_raw  
0   warmtepomp(en)  186513.0   186513.0  
1   warmtepomp(en)  186520.0   186520.0  
2   warmtepomp(en)  186529.0   186529.0  
3   warmtepomp(en)  186537.0   186537.0  
4   warmtepomp(en)  186544.0   186544.0  


### Maak de aggregaties

In [13]:
from aggregatie_dict import AGG_REGELS

In [14]:
# Map aggregatieregels (default 'mean' als niet gespecificeerd)
df_long['agg_methode'] = df_long['func_dp_nr'].map(AGG_REGELS).fillna('mean')

# Voer per aggregatiemethode grouping & aggregatie uit
parts = []
group_cols = ['timestamp', 'func_dp_nr', 'datapunt_naam', 'datapunt_systeem']
for methode in df_long['agg_methode'].unique():
    sub = df_long[df_long['agg_methode'] == methode]
    agg = sub.groupby(group_cols, as_index=False)['waarde'].agg(methode)
    parts.append(agg)

df_clean = pd.concat(parts, ignore_index=True).sort_values(by=['timestamp','func_dp_nr']).reset_index(drop=True)
display(df_clean.head())

,timestamp,func_dp_nr,datapunt_naam,datapunt_systeem,waarde
0,2026-03-09 00:00:00+00:00,1,totaal warmtegebruik warm,warmtepomp(en),186513.000000
1,2026-03-09 00:00:00+00:00,2,totaal debiet,warmtepomp(en),20546.530000
2,2026-03-09 00:00:00+00:00,3,status,warmtepomp(en),1.000000
3,2026-03-09 00:00:00+00:00,4,temperatuur vertrek,warmtepomp(en),39.038220
4,2026-03-09 00:00:00+00:00,6,temperatuur retour,warmtepomp(en),30.977285


#### Maak een tabel voor het trainen van het model

In [15]:
df_wide = (
    df_long
    .groupby(['timestamp', 'func_dp_nr'], as_index=False)['waarde']
    .mean()                                   # aggregeer dubbele metingen
    .pivot(index='timestamp', columns='func_dp_nr', values='waarde')
)

# verwijder de kolom-indexnaam en zet timestamp terug als kolom
df_wide.columns.name = None
df_wide = df_wide.reset_index().sort_values('timestamp').reset_index(drop=True)

# (optioneel) prefix feature-kolommen: f_1, f_2, ...
new_names = {}
for c in df_wide.columns:
    if c == 'timestamp':
        continue
    try:
        new_names[c] = f"f_{int(c)}"
    except Exception:
        new_names[c] = f"f_{c}"
df_wide = df_wide.rename(columns=new_names)
df_wide = df_wide.set_index('timestamp')

display(df_wide.head())
print("shape:", df_wide.shape)

,f_1,f_2,f_3,f_4,f_6,f_7,f_8,f_9,f_10,f_11,...,f_41,f_42,f_43,f_44,f_45,f_46,f_47,f_48,f_49,f_50
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-03-09 00:00:00+00:00,186513.0,20546.53,1.0,39.038220,30.977285,1.0,18.945000,0.000000,0.0,17.895657,...,20.309923,20.11093,21.25,19.573399,486.739307,50.0,19.666667,20.75229,486.869257,0.0
2026-03-09 00:10:00+00:00,186520.0,20547.30,1.0,39.877285,31.832268,1.0,23.594538,44.640636,1.0,22.222490,...,20.309923,20.11093,21.25,19.573399,488.981002,50.0,19.666667,20.75229,487.712690,0.0
2026-03-09 00:20:00+00:00,186529.0,20548.08,1.0,41.921010,34.104786,1.0,46.930084,44.640636,1.0,38.731342,...,20.309923,20.11093,21.25,19.573399,491.261212,50.0,19.666667,20.75229,485.292560,0.0
2026-03-09 00:30:00+00:00,186537.0,20548.94,1.0,43.412975,35.585346,1.0,43.235270,44.640636,0.0,41.083800,...,20.309923,20.11093,21.25,19.573399,494.401493,50.0,19.666667,20.75229,486.288320,0.0
2026-03-09 00:40:00+00:00,186544.0,20549.72,1.0,43.734930,35.924030,1.0,48.845810,44.640636,0.0,46.879260,...,20.309923,20.11093,21.25,19.573399,495.742685,50.0,19.666667,20.75229,490.234433,0.0


shape: (7774, 49)


### Dedecteer cumul en bereken diff

In [16]:
# Itereren over alle feature-kolommen in de brede dataset
for col in df_wide.columns:
    # Bereken het verschil met de vorige tijdstap (huidige waarde - vorige waarde)
    diffs = df_wide[col].diff()
    
    # Filter de NaN-waarden en de momenten waarop er geen verandering was (verschil == 0)
    veranderingen = diffs.dropna()
    actieve_veranderingen = veranderingen[veranderingen != 0]
    
    # Check of de variabele cumulatief is:
    # 1. Er moeten veranderingen zijn.
    # 2. Vrijwel alle (> 99%) actieve veranderingen moeten positief zijn (stijgende tellerstand).
    if len(actieve_veranderingen) > 0 and (actieve_veranderingen > 0).mean() >= 0.99:
        print(f"Cumulatieve trend gedetecteerd in {col}. Wordt omgezet naar periodiek verbruik.")
        
        # Vervang de kolom door het berekende verschil.
        # .clip(lower=0) is een robuuste veiligheidsmaatregel voor 'meter resets' 
        # (bijv. teller gaat van 9999 naar 0, wat een enorme negatieve piek zou geven).
        df_wide[col] = diffs.clip(lower=0)

# De eerste rij zal altijd NaN zijn na een .diff(), deze kunnen we opvullen met 0 of de rij weglaten
df_wide = df_wide.fillna(0)
# Laat de eerste rij weg zodat we enkel echte data hebben
df_wide = df_wide.iloc[1:]

Cumulatieve trend gedetecteerd in f_1. Wordt omgezet naar periodiek verbruik.
Cumulatieve trend gedetecteerd in f_2. Wordt omgezet naar periodiek verbruik.
Cumulatieve trend gedetecteerd in f_13. Wordt omgezet naar periodiek verbruik.
Cumulatieve trend gedetecteerd in f_14. Wordt omgezet naar periodiek verbruik.
Cumulatieve trend gedetecteerd in f_16. Wordt omgezet naar periodiek verbruik.
Cumulatieve trend gedetecteerd in f_17. Wordt omgezet naar periodiek verbruik.


## STAP 5 - Weerdata toevoegen

Website -> https://opendata.meteo.be/download  

> **Product**: `Automatic weather station (AWS)`  
> **Layer**: `aws_10min`  
> **Format**: `CSV`  

Omdat de dataset soms gaten vertoont voor het weerstation in Gent voegen we code toe om op basis van dichtst bijzijnde weerstations met data voor een tijdstap de kolom op te vullen zodat er geen missende data is.

In [17]:
# Laad de csv in
weer = pd.read_csv("raw/aws_10min.csv")

weather_cols = [
    'temp_dry_shelter_avg',
    'humidity_rel_shelter_avg',
    'wind_speed_10m',
    'wind_direction',
    'sun_int_avg'
]
cols = ['timestamp', 'the_geom'] + weather_cols
weer = weer[cols].copy()

weer['timestamp'] = pd.to_datetime(weer['timestamp'], utc=True).dt.round('10min')

weer.head()

,timestamp,the_geom,temp_dry_shelter_avg,humidity_rel_shelter_avg,wind_speed_10m,wind_direction,sun_int_avg
0,2026-03-01 00:00:00+00:00,POINT (51.325 4.364),5.702407,87.284715,4.256845,197.015318,0.0
1,2026-03-01 00:00:00+00:00,POINT (50.096 4.595),2.347000,99.300000,1.201000,102.700000,0.0
2,2026-03-01 00:00:00+00:00,POINT (51.221 5.027),4.224874,87.990694,1.380981,215.729628,0.0
3,2026-03-01 00:00:00+00:00,POINT (51.28 3.074),4.046508,NaN,NaN,NaN,0.0
4,2026-03-01 00:00:00+00:00,POINT (50.582 4.689),4.457751,78.706188,2.429730,195.661639,0.0


In [18]:
TARGET_GEOM = 'POINT (50.98 3.816)'  # jouw gewenste punt

def parse_point(s):
    # verwacht formaat: POINT (lat lon)
    m = re.match(r'POINT\s*\(([-0-9.]+)\s+([-0-9.]+)\)', str(s))
    if not m:
        return np.nan, np.nan
    return float(m.group(1)), float(m.group(2))

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

target_lat, target_lon = parse_point(TARGET_GEOM)

geo = weer[['the_geom']].drop_duplicates().copy()
geo[['lat', 'lon']] = geo['the_geom'].apply(lambda g: pd.Series(parse_point(g)))
geo = geo.dropna()
geo['dist_km'] = haversine_km(geo['lat'], geo['lon'], target_lat, target_lon)
geo = geo.sort_values('dist_km').reset_index(drop=True)

ordered_geoms = geo['the_geom'].tolist()

In [19]:
# 3) Fallback per timestamp: neem eerst dichtstbijzijnde niet-lege waarde
parts = []
for g in ordered_geoms:
    part = (
        weer[weer['the_geom'] == g][['timestamp'] + weather_cols]
        .drop_duplicates(subset='timestamp')
        .set_index('timestamp')
        .sort_index()
    )
    parts.append(part)

weer_fallback = parts[0].copy()
for p in parts[1:]:
    weer_fallback = weer_fallback.combine_first(p)

weer_fallback = weer_fallback.reset_index()

# (optioneel) diagnose
print("Dichtste geopunten:")
print(geo[['the_geom', 'dist_km']].head(5))

print("\nMissings na geo-fallback:")
print(weer_fallback[weather_cols].isna().sum())

Dichtste geopunten:
               the_geom    dist_km
0   POINT (50.98 3.816)   0.000000
1  POINT (50.797 4.358)  43.121681
2  POINT (50.904 3.122)  49.353585
3  POINT (51.075 4.525)  50.696974
4  POINT (51.325 4.364)  54.152581

Missings na geo-fallback:
temp_dry_shelter_avg        0
humidity_rel_shelter_avg    0
wind_speed_10m              0
wind_direction              0
sun_int_avg                 0
dtype: int64


### Voeg de kolommen toe als f_51, f_52, f_53, f_54, f_55

In [20]:
# Zorg timestamp kolom / index consistent
df_wide = df_wide.reset_index() if 'timestamp' not in df_wide.columns else df_wide.copy()
df_wide['timestamp'] = pd.to_datetime(df_wide['timestamp'], utc=True).dt.round('10min')

# Gebruik fallback-weerdata i.p.v. oorspronkelijke weer
weer_fallback['timestamp'] = pd.to_datetime(weer_fallback['timestamp'], utc=True).dt.round('10min')

# Selecteer & hernoem weer-kolommen naar features f_51..f_55
map_cols = {
    'temp_dry_shelter_avg': 'f_51',
    'humidity_rel_shelter_avg': 'f_52',
    'wind_speed_10m': 'f_53',
    'wind_direction': 'f_54',
    'sun_int_avg': 'f_55'
}

weer_sub = (
    weer_fallback[['timestamp'] + list(map_cols.keys())]
    .drop_duplicates(subset='timestamp')
    .rename(columns=map_cols)
    .sort_values('timestamp')
)

# Merge (left join zodat alle rijen van df_wide blijven)
df_merged = df_wide.merge(weer_sub, on='timestamp', how='left').sort_values('timestamp')

# Eventuele resterende gaten in weerfeatures opvullen (tijdinterpolatie + randen)
weather_features = ['f_51', 'f_52', 'f_53', 'f_54', 'f_55']
df_merged = df_merged.set_index('timestamp')
df_merged[weather_features] = (
    df_merged[weather_features]
    .interpolate(method='time')
    .ffill()
    .bfill()
)

# Controle: nu zouden deze 0 moeten zijn
missing_counts = df_merged[weather_features].isna().sum()
print("Ontbrekende waarden per weer-feature:\n", missing_counts)

# Extra check: hoeveel timestamps in df_wide missen in weer_sub en omgekeerd
df_t = pd.to_datetime(df_wide['timestamp'], utc=True).drop_duplicates()
weer_t = pd.to_datetime(weer_sub['timestamp'], utc=True).drop_duplicates()
print("timestamps in df_wide not in weer_sub:", (~df_t.isin(weer_t)).sum())
print("timestamps in weer_sub not in df_wide:", (~weer_t.isin(df_t)).sum())

# resultaat tonen
display(df_merged.head())

Ontbrekende waarden per weer-feature:
 f_51    0
f_52    0
f_53    0
f_54    0
f_55    0
dtype: int64
timestamps in df_wide not in weer_sub: 0
timestamps in weer_sub not in df_wide: 1209


,f_1,f_2,f_3,f_4,f_6,f_7,f_8,f_9,f_10,f_11,...,f_46,f_47,f_48,f_49,f_50,f_51,f_52,f_53,f_54,f_55
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-03-09 00:10:00+00:00,7.0,0.77,1.0,39.877285,31.832268,1.0,23.594538,44.640636,1.0,22.222490,...,50.0,19.666667,20.75229,487.712690,0.0,3.851797,99.589400,0.749325,86.946641,0.0
2026-03-09 00:20:00+00:00,9.0,0.78,1.0,41.921010,34.104786,1.0,46.930084,44.640636,1.0,38.731342,...,50.0,19.666667,20.75229,485.292560,0.0,3.903977,100.000000,0.559565,45.771835,0.0
2026-03-09 00:30:00+00:00,8.0,0.86,1.0,43.412975,35.585346,1.0,43.235270,44.640636,0.0,41.083800,...,50.0,19.666667,20.75229,486.288320,0.0,4.244891,99.196013,0.322963,91.228401,0.0
2026-03-09 00:40:00+00:00,7.0,0.78,1.0,43.734930,35.924030,1.0,48.845810,44.640636,0.0,46.879260,...,50.0,19.666667,20.75229,490.234433,0.0,4.448537,98.800579,0.300000,323.796930,0.0
2026-03-09 00:50:00+00:00,9.0,0.78,1.0,44.037724,36.229050,1.0,44.068630,44.640636,0.0,44.223957,...,50.0,19.666667,20.75229,488.682280,0.0,4.748848,99.259927,0.345344,204.933704,0.0


## Sla de dataset op voor verdere analyse

In [21]:
df_merged.to_csv(f'preprocessed/{GEBOUWNAAM}.csv', index=True, date_format='%Y-%m-%dT%H:%M:%SZ')